# 02 — PDAL Pipeline: Ground Classification and Height Normalization

## What this notebook does

Runs the PDAL pipeline (`pipeline/pdal_pipeline.json`) on the raw point cloud. Two operations are applied:

1. **Ground classification** — `filters.csf` assigns ASPRS class 2 to ground returns
2. **Height normalization** — `filters.hag_nn` adds a `HeightAboveGround` dimension to every point

---

## Inputs

- `data/raw/socal_sespe_32611.laz` — raw tile from Notebook 01
- `pipeline/pdal_pipeline.json` — pipeline definition (path substitution handled by `run_pipeline`)

## Outputs

- `data/processed/socal_sespe_normalized.laz` — classified and HAG-normalized point cloud

## Functions called

- `src.pdal_runner.check_pdal_available()`
- `src.pdal_runner.run_pipeline(pipeline_json_path, input_laz, output_laz)`

---

## What to confirm before moving to Notebook 03

- Output LAZ exists at `data/processed/socal_sespe_normalized.laz`
- Re-inspect classification distribution — class 2 (ground) points should now be present
- If class 2 count is zero or suspiciously low, revisit CSF parameters (`resolution`, `rigidness`, `slope_smooth`) in the pipeline JSON

## CSF parameter guidance

Your terrain is chaparral foothill — sloped but not extreme. Start with:
- `resolution`: 0.5 or 1.0
- `rigidness`: 2 (default)
- `slope_smooth`: true

Adjust after inspecting results visually.


In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent  # assumes notebook is in notebooks/
pipeline_json_path = ROOT / "pipeline" / "pdal_pipeline.json"
input_data_path = ROOT / "data" / "raw" / "socal_sespe_32611.laz"
output_data_path = ROOT / "data" / "processed" / "socal_sespe_32611_normalized.laz"

In [7]:
import sys
sys.path.append(str(ROOT))

from src.pdal_runner import check_pdal_available, run_pipeline

In [8]:
check_pdal_available()

'pdal 2.10.0 (git-version: Release)'

In [9]:
# run_pipeline(
#     pipeline_json_path=pipeline_json_path,
#     input_laz=input_data_path,
#     output_laz=output_data_path
# )

In [18]:
from src.data_utils import load_laz

raw_las = load_laz(input_data_path)
normalized_las = load_laz(output_data_path)

raw_df = pd.DataFrame({
    "x": np.array(raw_las.x).flatten(),
    "y": np.array(raw_las.y).flatten(),
    "z": np.array(raw_las.z).flatten(),
    "classification": np.array(raw_las.classification).flatten(),
    "return_number": np.array(raw_las.return_number).flatten(),
    "intensity": np.array(raw_las.intensity).flatten(),
})

normal_df = pd.DataFrame({
    "x": np.array(normalized_las.x).flatten(),
    "y": np.array(normalized_las.y).flatten(),
    "z": np.array(normalized_las.z).flatten(),
    "classification": np.array(normalized_las.classification).flatten(),
    "return_number": np.array(normalized_las.return_number).flatten(),
    "intensity": np.array(normalized_las.intensity).flatten(),
})

print(f"Raw DataFrame classification counts:\n{raw_df.value_counts('classification')}")
print(f"Normalized DataFrame classification counts:\n{normal_df.value_counts('classification')}")

Raw DataFrame classification counts:
classification
1     1621602
2      384551
9         139
7          12
10          4
Name: count, dtype: int64
Normalized DataFrame classification counts:
classification
1    1349470
2     656838
Name: count, dtype: int64
